# Voice leading minimal par affectation

**Navigation** : [Index](README.md) | [Version Python](App-21-VoiceLeading.ipynb) | [App-20 Sudoku](App-20-SudokuBenchmark-Python.ipynb)

**Objectifs** : formuler le mouvement des voix comme affectation ; comparer glouton et optimum sur 36 transitions ; visualiser deux géométries ; auditer et réparer les parallèles de Fux. **Prérequis** : matrices de coûts, permutations et notes MIDI. **Durée estimée** : 90 minutes.

Hommage au mathématicien et musicien James R. Munkres : l'affectation relie ici l'optimisation à la conduite des voix. Le solveur industriel OR-Tools `LinearSumAssignment` traite le coût linéaire ; CP-SAT traite les contraintes couplant des paires de voix. Le Python utilise SciPy (Jonker–Volgenant modifié) pour l'affectation.

In [1]:
#r "nuget: Google.OrTools, 9.15.6755"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using System.Text;
using System.Diagnostics;
using Google.OrTools.Graph;
using Google.OrTools.Sat;
CultureInfo.DefaultThreadCurrentCulture=CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription} ; OR-Tools {typeof(CpModel).Assembly.GetName().Version}");

Installed Packages Google.OrTools, 9.15.6755

Runtime .NET 10.0.14 ; OR-Tools 9.15.6755.0


## 1. Du mouvement musical à la matrice de coûts

Pour chaque voix source $i$ et note cible $j$, la distance chromatique vaut $|s_i-t_j|$. Sur la ligne des quintes, on utilise les positions signées dans $[-5,6]$ et leur distance linéaire, pas une distance circulaire. Un accord cible est une permutation des notes par voix.

In [2]:
string[] Notes={"C","C#","D","D#","E","F","F#","G","G#","A","A#","B"};string Nom(int midi)=>$"{Notes[midi%12]}{midi/12-1}";int Quinte(int note){int v=(note%12*7)%12;return v<=6?v:v-12;}
int[,] Cout(int[] s,int[] t,bool quintes=false){var m=new int[s.Length,t.Length];for(int i=0;i<s.Length;i++)for(int j=0;j<t.Length;j++)m[i,j]=quintes?Math.Abs(Quinte(s[i])-Quinte(t[j])):Math.Abs(s[i]-t[j]);return m;}
var accords=new Dictionary<string,int[]>{{"C",new[]{60,64,67}},{"C/1",new[]{64,67,72}},{"C/2",new[]{55,60,64}},{"F",new[]{53,57,60}},{"F/1",new[]{57,60,65}},{"F/2",new[]{60,65,69}},{"G",new[]{55,59,62}},{"G/1",new[]{59,62,67}},{"G/2",new[]{62,67,71}},{"Dm",new[]{62,65,69}},{"Dm/1",new[]{53,57,62}},{"Dm/2",new[]{57,62,65}},{"Am",new[]{57,60,64}},{"Am/1",new[]{60,64,69}},{"Am/2",new[]{52,57,60}}};
var S=accords["C"];var T=accords["F/1"];var chrom=Cout(S,T);var fifth=Cout(S,T,true);foreach(var(nom,m) in new[]{("Chromatique",chrom),("Quintes",fifth)}){Console.WriteLine(nom+" :");for(int i=0;i<S.Length;i++)Console.WriteLine($"{Nom(S[i]),3} : "+string.Join(" ",Enumerable.Range(0,T.Length).Select(j=>m[i,j].ToString().PadLeft(2))));}Console.WriteLine($"Positions : C={Quinte(60)}, G={Quinte(67)}, E={Quinte(64)}, F={Quinte(65)}");

Chromatique :


 C4 :  3  0  5


 E4 :  7  4  1


 G4 : 10  7  2


Quintes :


 C4 :  3  0  1


 E4 :  1  4  5


 G4 :  2  1  2


Positions : C=0, G=1, E=4, F=-1


### Lecture du résultat

Les deux matrices expriment des choix différents : un mouvement court en demi-tons n'est pas forcément court sur la ligne des quintes. Les positions sont des représentants signés fixés, sans raccourci circulaire.

## 2. Affectation optimale et glouton

Le moteur natif résout l'affectation. Pour départager les minima de coût, nous énumérons ensuite les $n!$ permutations et retenons celle qui minimise le plus grand mouvement ; ce départage pédagogique est factoriel, même si le problème d'affectation possède des algorithmes polynomiaux.

In [3]:
IEnumerable<int[]> Permutations(int n){IEnumerable<int[]> Rec(List<int> p){if(p.Count==n){yield return p.ToArray();yield break;}for(int j=0;j<n;j++)if(!p.Contains(j)){p.Add(j);foreach(var a in Rec(p))yield return a;p.RemoveAt(p.Count-1);}}return Rec(new List<int>());}
int Somme(int[,] m,int[] p)=>Enumerable.Range(0,p.Length).Sum(i=>m[i,p[i]]);
(int[],int) Optimum(int[,] m){int n=m.GetLength(0);if(m.GetLength(1)!=n)throw new ArgumentException("Matrice carrée requise");using var solver=new LinearSumAssignment();for(int i=0;i<n;i++)for(int j=0;j<n;j++)solver.AddArcWithCost(i,j,m[i,j]);var status=solver.Solve();if(status!=LinearSumAssignment.Status.OPTIMAL)throw new Exception($"Affectation : {status}");int cost=checked((int)solver.OptimalCost());var perm=Permutations(n).Where(p=>Somme(m,p)==cost).OrderBy(p=>Enumerable.Range(0,n).Max(i=>m[i,p[i]])).First();return(perm,cost);}
(int[],int) Glouton(int[,] m){int n=m.GetLength(0);var used=new HashSet<int>();var p=new int[n];for(int i=0;i<n;i++){int j=Enumerable.Range(0,n).Where(x=>!used.Contains(x)).OrderBy(x=>m[i,x]).ThenBy(x=>x).First();p[i]=j;used.Add(j);}return(p,Somme(m,p));}
var (gp,gcost)=Glouton(chrom);var (op,ocost)=Optimum(chrom);var (fq,fcost)=Optimum(fifth);Console.WriteLine($"Glouton : {gcost} demi-tons ; optimal : {ocost} demi-tons ; quintes : {fcost} positions");for(int i=0;i<S.Length;i++)Console.WriteLine($"{Nom(S[i])} → {Nom(T[op[i]])} : {chrom[i,op[i]]} demi-tons, {fifth[i,op[i]]} positions");

Glouton : 11 demi-tons ; optimal : 9 demi-tons ; quintes : 3 positions


C4 → A3 : 3 demi-tons, 3 positions


E4 → C4 : 4 demi-tons, 4 positions


G4 → F4 : 2 demi-tons, 2 positions


### Lecture du résultat

Le glouton prend d'abord C4→C4 et laisse un saut de G4 vers A3. L'affectation optimale répartit les mouvements ; G4→F4 descend de deux demi-tons. La même affectation se projette ici sur 3, 4 et 2 positions de quinte respectivement.

### Exercice 1 — Comparer les métriques

Retournez les deux permutations optimales et un indicateur d'égalité. La garde vérifie des matrices non triviales et une permutation complète. Ici le coût glouton vaut 11 demi-tons contre 9 pour l’optimum, et 3 positions pour la métrique des quintes.

In [4]:
(int[] chrom,int[] quintes,bool identiques)? ComparerMetriques(int[,] c,int[,] q){return null;} // TODO : appeler Optimum pour chaque métrique
bool VerifierMetriques((int[] chrom,int[] quintes,bool identiques)? r,int[,] c,int[,] q){if(!r.HasValue)return false;var v=r.Value;int n=c.GetLength(0);if(v.chrom.Length!=n||v.quintes.Length!=n||!v.chrom.OrderBy(x=>x).SequenceEqual(Enumerable.Range(0,n))||!v.quintes.OrderBy(x=>x).SequenceEqual(Enumerable.Range(0,n))||Somme(c,v.chrom)!=Optimum(c).Item2||Somme(q,v.quintes)!=Optimum(q).Item2||v.identiques!=v.chrom.SequenceEqual(v.quintes))throw new Exception("Comparaison des métriques incorrecte");return true;}
var cm1=ComparerMetriques(chrom,fifth);var cm2=ComparerMetriques(new int[,]{{0,4},{3,0}},new int[,]{{3,0},{0,3}});bool m1=VerifierMetriques(cm1,chrom,fifth),m2=VerifierMetriques(cm2,new int[,]{{0,4},{3,0}},new int[,]{{3,0},{0,3}});Console.WriteLine(m1&&m2?$"Mêmes permutations sur C→F/1 : {cm1.Value.identiques}":"Exercice 1 à compléter : métriques");

Exercice 1 à compléter : métriques


## 3. Balayage des renversements

Quatre transitions harmoniques, trois renversements de chaque côté : $4\times3\times3=36$ cas. Les différences entre glouton et optimum sont calculées sur chaque matrice.

In [5]:
var familles=new[]{("Dm","G","ii-V"),("G","C","V-I"),("F","G","IV-V"),("C","Am","I-vi")};var lignes=new List<(string label,string src,string tgt,int[] s,int[] t,int g,int o)>();foreach(var(sf,tf,label) in familles)foreach(var sk in accords.Keys.Where(k=>k==sf||k.StartsWith(sf+"/")))foreach(var tk in accords.Keys.Where(k=>k==tf||k.StartsWith(tf+"/"))){var a=accords[sk];var b=accords[tk];var mat=Cout(a,b);lignes.Add((label,sk,tk,a,b,Glouton(mat).Item2,Optimum(mat).Item2));}
var pires=lignes.Where(r=>r.g>r.o).ToArray();Console.WriteLine($"{lignes.Count} transitions ; glouton sous-optimal {pires.Length}, égal {lignes.Count-pires.Length} ; écart moyen {pires.Average(r=>r.g-r.o):F2}, maximum {pires.Max(r=>r.g-r.o)} demi-tons");foreach(var r in lignes.Take(12))Console.WriteLine($"{r.label,-5} {r.src,-4} → {r.tgt,-4} : glouton {r.g}, optimum {r.o}");

36 transitions ; glouton sous-optimal 5, égal 31 ; écart moyen 4.40, maximum 6 demi-tons


ii-V  Dm   → G    : glouton 20, optimum 20


ii-V  Dm   → G/1  : glouton 12, optimum 8


ii-V  Dm   → G/2  : glouton 4, optimum 4


ii-V  Dm/1 → G    : glouton 4, optimum 4


ii-V  Dm/1 → G/1  : glouton 16, optimum 16


... (3 outputs left out)

ii-V  Dm/2 → G/2  : glouton 16, optimum 16


V-I   G    → C    : glouton 15, optimum 15


V-I   G    → C/1  : glouton 27, optimum 27


V-I   G    → C/2  : glouton 3, optimum 3


### Lecture du résultat

La différence n'est pas universelle : plusieurs renversements sont déjà optimaux en glouton. Les cas perdants montrent pourquoi un choix local ne suffit pas à garantir le coût global.

### Exercice 2 — Pénalité de croisement approchée

Construisez $c_{ij}=|s_i-t_j|+\lambda\sum_{k\ne i}1[s_i<s_k\land t_j>t_k]$. Les cibles des autres voix sont figées à leur indice : c'est un **proxy linéaire**, pas l'optimisation exacte des croisements de la permutation. Sur ce panel, le glouton est sous-optimal 5 fois sur 36 ; l’écart moyen sur ces cinq cas vaut 4,40 demi-tons.

In [6]:
int[,] Penalite(int[] source,int[] cible,int lambda){return null;} // TODO : coût chromatique + proxy indiqué
bool VerifierPenalite(int[,] r,int[] s,int[] t,int lam){if(r is null)return false;var c=Cout(s,t);if(r.GetLength(0)!=s.Length||r.GetLength(1)!=t.Length)throw new Exception("Dimensions incorrectes");for(int i=0;i<s.Length;i++)for(int j=0;j<t.Length;j++){int p=0;for(int k=0;k<s.Length;k++)if(k!=i&&s[i]<s[k]&&t[j]>t[k])p++;if(r[i,j]!=c[i,j]+lam*p)throw new Exception("Pénalité incorrecte");}return true;}
var pe1=Penalite(S,T,3);var pe2=Penalite(new[]{55,60,68},new[]{64,58,71},4);bool p1=VerifierPenalite(pe1,S,T,3),p2=VerifierPenalite(pe2,new[]{55,60,68},new[]{64,58,71},4);if(p1&&p2){foreach(int lam in new[]{0,3,6}){var mat=Penalite(S,T,lam);if(!VerifierPenalite(mat,S,T,lam)){Console.WriteLine($"Proxy incomplet pour λ={lam}");break;}var pp=Optimum(mat);int inversions=0;for(int i=0;i<pp.Item1.Length;i++)for(int k=i+1;k<pp.Item1.Length;k++)if(pp.Item1[i]>pp.Item1[k])inversions++;Console.WriteLine($"λ={lam} : coût proxy {pp.Item2}, coût chromatique {Somme(chrom,pp.Item1)}, inversions {inversions}");}}else Console.WriteLine("Exercice 2 à compléter : proxy de croisement");

Exercice 2 à compléter : proxy de croisement


## 4. Visualiser les trajectoires

La première figure compare les mouvements MIDI du glouton et de l'optimum. La seconde projette la même affectation optimale sur la ligne des quintes, avec les positions signées des notes.

In [7]:
record Svg(string Content);
Microsoft.DotNet.Interactive.Formatting.Formatter.Register<Svg>((v,w)=>w.Write(v.Content),"text/html");
string[] couleurs={"#1f77b4","#d62728","#2ca02c"};var svg=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='760' height='410' viewBox='0 0 760 410'><rect width='760' height='410' fill='white'/><text x='380' y='25' text-anchor='middle'>C → F/1 : trajectoires des voix</text><text x='18' y='200' transform='rotate(-90 18 200)' text-anchor='middle'>hauteur MIDI</text>");double Y(int n)=>330-(n-55)*14;for(int panel=0;panel<2;panel++){int x=75+panel*350;svg.Append($"<text x='{x+125}' y='50' text-anchor='middle'>{(panel==0?"Glouton":"Optimal")}</text><line x1='{x}' y1='330' x2='{x+250}' y2='330' stroke='black'/><line x1='{x}' y1='70' x2='{x}' y2='330' stroke='black'/>");for(int note=56;note<=72;note+=4)svg.Append($"<line x1='{x}' y1='{Y(note):F1}' x2='{x+250}' y2='{Y(note):F1}' stroke='#ddd'/><text x='{x-8}' y='{Y(note)+4:F1}' text-anchor='end'>{note}</text>");int[] perm=panel==0?gp:op;for(int i=0;i<3;i++)svg.Append($"<line x1='{x+35}' y1='{Y(S[i]):F1}' x2='{x+210}' y2='{Y(T[perm[i]]):F1}' stroke='{couleurs[i]}' stroke-width='3'/><circle cx='{x+35}' cy='{Y(S[i]):F1}' r='4' fill='{couleurs[i]}'/><circle cx='{x+210}' cy='{Y(T[perm[i]]):F1}' r='4' fill='{couleurs[i]}'/>");svg.Append($"<text x='{x+35}' y='350' text-anchor='middle'>départ</text><text x='{x+210}' y='350' text-anchor='middle'>arrivée</text>");}for(int i=0;i<3;i++)svg.Append($"<rect x='{100+i*200}' y='375' width='12' height='12' fill='{couleurs[i]}'/><text x='{118+i*200}' y='386'>{Nom(S[i])}</text>");svg.Append("</svg>");display(new Svg(svg.ToString()));Console.WriteLine("Trajectoires chromatiques : glouton et optimum.");

C → F/1 : trajectoires des voix hauteur MIDI Glouton 56 60 64 68 72 départ arrivée Optimal 56 60 64 68 72 départ arrivée C4 E4 G4

Trajectoires chromatiques : glouton et optimum.


### Lecture de la première figure

Chaque couleur suit une voix, et les points indiquent ses hauteurs de départ et d'arrivée. Le grand saut glouton apparaît sur la voix issue de G4.

In [8]:
var line=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='760' height='280' viewBox='0 0 760 280'><rect width='760' height='280' fill='white'/><text x='380' y='25' text-anchor='middle'>Affectation chromatique sur la ligne des quintes</text><line x1='80' y1='140' x2='680' y2='140' stroke='#aaa' stroke-width='5'/>");for(int k=-1;k<=5;k++){int x=180+85*k;string label=k switch{-1=>"F",0=>"C",1=>"G",2=>"D",3=>"A",4=>"E",_=>"B"};line.Append($"<circle cx='{x}' cy='140' r='13' fill='#666'/><text x='{x}' y='144' text-anchor='middle' fill='white'>{label}</text><text x='{x}' y='165' text-anchor='middle'>{k}</text>");}for(int i=0;i<3;i++){int x1=180+85*Quinte(S[i]),x2=180+85*Quinte(T[op[i]]);line.Append($"<line x1='{x1}' y1='90' x2='{x2}' y2='190' stroke='{couleurs[i]}' stroke-width='3'/><circle cx='{x1}' cy='90' r='4' fill='{couleurs[i]}'/><circle cx='{x2}' cy='190' r='4' fill='{couleurs[i]}'/><text x='{x1}' y='77' text-anchor='middle' fill='{couleurs[i]}'>{Nom(S[i])}</text><text x='{x2}' y='210' text-anchor='middle' fill='{couleurs[i]}'>{Nom(T[op[i]])}</text>");}line.Append("<text x='380' y='250' text-anchor='middle'>positions signées sur la ligne des quintes</text></svg>");display(new Svg(line.ToString()));Console.WriteLine("Projection : départ en haut, arrivée en bas.");

Affectation chromatique sur la ligne des quintes F -1 C 0 G 1 D 2 A 3 E 4 B 5 C4 A3 E4 C4 G4 F4 positions signées sur la ligne des quintes

Projection : départ en haut, arrivée en bas.


### Lecture de la seconde figure

La projection conserve les identités des voix. Elle montre les distances sur les représentants signés : E→C parcourt 4 positions, C→A 3 et G→F 2.

## 5. Une progression autonome de six accords

L'exemple Dm–G–C–F–G–C est fourni localement. Chaque voix garde son indice d'une transition à l'autre : `nouveau[i] = cible[permutation[i]]`. Une intégration à un générateur de musique exige une conversion explicite des accords et registres ; aucun MIDI n'est exporté ici.

In [9]:
(List<int[]>,int) Revoicer(string[] progression,bool optimal){var voix=accords[progression[0]].ToArray();var result=new List<int[]>{voix};int total=0;foreach(string name in progression.Skip(1)){var cible=accords[name];var m=Cout(voix,cible);var (p,c)=optimal?Optimum(m):Glouton(m);voix=Enumerable.Range(0,p.Length).Select(i=>cible[p[i]]).ToArray();result.Add(voix);total+=c;}return(result,total);}
var progression=new[]{"Dm","G","C","F","G","C"};var (vg,cg)=Revoicer(progression,false);var (vo,co)=Revoicer(progression,true);Console.WriteLine("Progression : "+string.Join(" – ",progression));Console.WriteLine("Glouton : "+string.Join(" | ",vg.Select(v=>string.Join(" ",v.Select(Nom))))+$" ; coût {cg}");Console.WriteLine("Optimal : "+string.Join(" | ",vo.Select(v=>string.Join(" ",v.Select(Nom))))+$" ; coût {co}");

Progression : Dm – G – C – F – G – C


Glouton : D4 F4 A4 | D4 B3 G3 | C4 E4 G4 | C4 A3 F3 | B3 G3 D4 | C4 E4 G4 ; coût 87


Optimal : D4 F4 A4 | G3 B3 D4 | C4 E4 G4 | F3 A3 C4 | G3 B3 D4 | C4 E4 G4 ; coût 77


### Lecture du résultat

Le coût cumulé est la somme des cinq transitions successives ; une affectation optimale à chaque pas n'optimise pas nécessairement toute la progression simultanément.

### Exercice 3 — Fixer la basse

La voix la plus grave doit prendre la cible la plus grave. Résolvez le sous-problème sur les autres voix et réassemblez la permutation. Dans cette exécution, les coûts cumulés sont 87 demi-tons pour le glouton et 77 pour le choix optimal à chaque étape.

In [10]:
(int[] permutation,int cout)? BasseFixee(int[] source,int[] cible){return null;} // TODO : enlever la ligne et la colonne, puis appeler Optimum
bool VerifierBasse((int[] permutation,int cout)? r,int[] s,int[] t){if(!r.HasValue)return false;var (p,c)=r.Value;int n=s.Length,bi=Enumerable.Range(0,n).OrderBy(i=>s[i]).First(),bj=Enumerable.Range(0,n).OrderBy(i=>t[i]).First();var m=Cout(s,t);if(p.Length!=n||!p.OrderBy(i=>i).SequenceEqual(Enumerable.Range(0,n))||p[bi]!=bj||c!=Somme(m,p)||c!=Permutations(n).Where(x=>x[bi]==bj).Min(x=>Somme(m,x)))throw new Exception("Basse contrainte incorrecte");return true;}
var ba1=BasseFixee(S,T);var bs=new[]{67,60,64};var bt=new[]{65,57,60};var ba2=BasseFixee(bs,bt);bool b1=VerifierBasse(ba1,S,T),b2=VerifierBasse(ba2,bs,bt);Console.WriteLine(b1&&b2?$"Basse imposée : coût {ba1.Value.cout} demi-tons":"Exercice 3 à compléter : basse fixe");

Exercice 3 à compléter : basse fixe


## 6. Audit de Fux et réparation CP-SAT

Un parallèle interdit conserve un intervalle dirigé de quinte ou d'octave avec un **déplacement non nul**. Deux voix immobiles conservent certes leur intervalle, mais ne réalisent pas de mouvement parallèle. L'encodage CP-SAT interdit les couples de décisions correspondants et peut ajouter le non-croisement.

In [11]:
List<(int,int,int)> Paralleles(int[] s,int[] t,int[] p){var v=new List<(int,int,int)>();for(int i=0;i<s.Length;i++)for(int k=i+1;k<s.Length;k++){int d=s[k]-s[i],fin=t[p[k]]-t[p[i]];if(d!=0&&d==fin&&(d%12==0||d%12==7)&&t[p[i]]!=s[i])v.Add((i,k,d));}return v;}
(int[],int?,string) Reparer(int[] s,int[] t,bool nonCroisement=false){int n=s.Length;var mat=Cout(s,t);var model=new CpModel();var x=new BoolVar[n,n];for(int i=0;i<n;i++)for(int j=0;j<n;j++)x[i,j]=model.NewBoolVar($"x{i}_{j}");for(int i=0;i<n;i++)model.AddExactlyOne(Enumerable.Range(0,n).Select(j=>x[i,j]));for(int j=0;j<n;j++)model.AddExactlyOne(Enumerable.Range(0,n).Select(i=>x[i,j]));for(int i=0;i<n;i++)for(int k=i+1;k<n;k++)for(int j=0;j<n;j++)for(int l=0;l<n;l++){int d=s[k]-s[i];bool parallele=d!=0&&(d%12==0||d%12==7)&&t[l]-t[j]==d&&t[j]!=s[i];bool croise=nonCroisement&&((s[i]<s[k])!=(t[j]<t[l]));if(parallele||croise)model.AddBoolOr(new ILiteral[]{x[i,j].Not(),x[k,l].Not()});}model.Minimize(LinearExpr.Sum(from i in Enumerable.Range(0,n) from j in Enumerable.Range(0,n) select mat[i,j]*x[i,j]));var solver=new CpSolver{StringParameters="max_time_in_seconds:10"};var st=solver.Solve(model);if(st!=CpSolverStatus.Optimal&&st!=CpSolverStatus.Feasible)return(null,null,st.ToString());var p=Enumerable.Range(0,n).Select(i=>Enumerable.Range(0,n).Single(j=>solver.Value(x[i,j])==1)).ToArray();if(Paralleles(s,t,p).Count>0)throw new Exception("Réparation Fux incorrecte");return(p,Somme(mat,p),st.ToString());}
var viol=lignes.Where(r=>Paralleles(r.s,r.t,Optimum(Cout(r.s,r.t)).Item1).Count>0).ToArray();Console.WriteLine($"Audit : {viol.Length} transitions fautives sur {lignes.Count}");foreach(var r in viol){var fix=Reparer(r.s,r.t);Console.WriteLine($"{r.label} {r.src}→{r.tgt} : {r.o} → {fix.Item2}, statut {fix.Item3}");}var vi=Reparer(accords["G"],accords["C"],true);Console.WriteLine($"V-I avec Fux et non-croisement : {vi.Item3}");var immobiles=Paralleles(new[]{60,67},new[]{60,67},new[]{0,1});if(immobiles.Count!=0)throw new Exception("Voix immobiles classées parallèles");Console.WriteLine("Deux voix immobiles : aucun mouvement parallèle.");
var chrono=Stopwatch.StartNew();Optimum(Cout(accords["G"],accords["C"]));chrono.Stop();double tA=chrono.Elapsed.TotalMilliseconds;chrono.Restart();Reparer(accords["G"],accords["C"]);chrono.Stop();Console.WriteLine($"Affectation avec départage : {tA:F3} ms ; CP-SAT avec Fux : {chrono.Elapsed.TotalMilliseconds:F3} ms (mesures locales)");

Audit : 4 transitions fautives sur 36


ii-V Dm→G : 20 → 20, statut Optimal


V-I G→C : 15 → 15, statut Optimal


IV-V F→G : 6 → 8, statut Optimal


I-vi C→Am : 10 → 10, statut Optimal


V-I avec Fux et non-croisement : Infeasible


Deux voix immobiles : aucun mouvement parallèle.


Affectation avec départage : 0.062 ms ; CP-SAT avec Fux : 2.732 ms (mesures locales)


### Lecture du résultat

CP-SAT optimise le même coût sous des contraintes supplémentaires ; le statut indique si une solution est prouvée ou si les contraintes sont infaisables. L'audit des 36 transitions vérifie chaque optimum indépendamment.

### Exercice 4 — Audit du mouvement direct

Détectez les paires qui bougent toutes deux dans le même sens non nul et arrivent sur une classe d'intervalle 0 ou 7 alors que l'intervalle initial est d'une autre classe. Ensuite, auditez les 36 transitions et la réparation V-I. Quatre des 36 optima présentent des parallèles ; CP-SAT les répare, et la combinaison Fux + non-croisement est infaisable pour V-I fondamental.

In [12]:
List<(int,int)> Directs(int[] s,int[] t,int[] p){return null;} // TODO : parcourir les paires de voix
bool VerifierDirects(List<(int,int)> r,int[] s,int[] t,int[] p){if(r is null)return false;var attendu=new List<(int,int)>();for(int i=0;i<s.Length;i++)for(int k=i+1;k<s.Length;k++){int d1=t[p[i]]-s[i],d2=t[p[k]]-s[k];int debut=((s[k]-s[i])%12+12)%12,fin=((t[p[k]]-t[p[i]])%12+12)%12;if(d1!=0&&Math.Sign(d1)==Math.Sign(d2)&&(fin==0||fin==7)&&debut!=0&&debut!=7)attendu.Add((i,k));}if(r.Count!=attendu.Count||!r.OrderBy(x=>x).SequenceEqual(attendu.OrderBy(x=>x)))throw new Exception("Audit direct incorrect");return true;}
var da=new[]{60,64};var db=new[]{62,69};var dp=new[]{0,1};var dc=new[]{60,67};var dd=new[]{60,67};var rda=Directs(da,db,dp);var rdb=Directs(dc,dd,dp);bool d1=VerifierDirects(rda,da,db,dp),d2=VerifierDirects(rdb,dc,dd,dp);if(!(d1&&d2))Console.WriteLine("Exercice 4 à compléter : mouvement direct");else{int total=0;bool incomplet=false;foreach(var r in lignes){var perm=Optimum(Cout(r.s,r.t)).Item1;var v=Directs(r.s,r.t,perm);if(VerifierDirects(v,r.s,r.t,perm))total+=v.Count;else incomplet=true;}var fix=Reparer(accords["G"],accords["C"]);if(fix.Item1!=null){var v=Directs(accords["G"],accords["C"],fix.Item1);if(VerifierDirects(v,accords["G"],accords["C"],fix.Item1))Console.WriteLine($"Réparation V-I : {v.Count} mouvements directs");else incomplet=true;}Console.WriteLine(incomplet?"Audit incomplet":"36 transitions auditées : "+total+" mouvements directs");}

Exercice 4 à compléter : mouvement direct


## 7. Synthèse et exercices

Les quatre exercices répartis explorent métriques, proxy de croisement, basse contrainte et mouvement direct. Le projet étudiant H1 de composition musicale assistée par contraintes a inspiré l'encodage des interdits Fux ; le projet H1_V2 traite quant à lui la génération mélodique avec contraintes souples.

## À retenir

- L'affectation minimise un coût global ; le glouton peut perdre sur certains renversements.
- L'identité de chaque voix doit être conservée dans une progression.
- Les règles de contrepoint couplent des voix et se formulent en contraintes ; une paire immobile ne constitue pas un parallèle.
- Le départage des optima par énumération est factoriel ; la métrique des quintes utilisée ici est linéaire sur des représentants signés.

## Vérifiez votre compréhension

1. Pourquoi le glouton peut-il perdre ?

<details><summary>Réponse</summary>
Une voix peut prendre une cible dont une autre a davantage besoin ; l'affectation minimise le coût de toutes les voix ensemble.
</details>

2. Pourquoi CP-SAT intervient-il ?

<details><summary>Réponse</summary>
Les interdits Fux concernent des couples de voix, donc la permutation entière, et ne se résument pas à un coût indépendant par paire source-cible.
</details>

3. Une paire immobile forme-t-elle un parallèle ?

<details><summary>Réponse</summary>
Non : l'intervalle est conservé, mais les deux déplacements sont nuls.
</details>

## Pour aller plus loin

[Kuhn (1955)](https://doi.org/10.1002/nav.3800020109), Munkres (1957), Tymoczko, *A Geometry of Music* (2011) ; [App-22 — sélection d'algorithmes](App-22-AlgorithmSelection-Python.ipynb) ; [CSP-3](../Part2-CSP/CSP-3-Advanced.ipynb).